In [ ]:
from pathlib import Path
from tempfile import mkdtemp
from urllib.parse import urlsplit
import importlib
import json
import os
import sys
import tarfile

import boto3
import pandas as pd

MODEL_ID = "usecase2"

# Every input comes from S3; no existing local data directory is used.
S3_INPUTS = {
    "data": "s3://src1007cced-243788878595-us-east-1/wsf-mlops/src1007cced-i1030a/models/usecase2/runs/cd97jkwzjyqo/data/raw/data.csv",
    "selection": "s3://src1007cced-243788878595-us-east-1/wsf-mlops/src1007cced-i1030a/models/usecase2/runs/cd97jkwzjyqo/data/selection/selected_features.csv",
    "model": "s3://src1007cced-243788878595-us-east-1/wsf-mlops/src1007cced-i1030a/models/usecase2/runs/cd97jkwzjyqo/model/application/model.tar.gz",
    "predictions": "s3://src1007cced-243788878595-us-east-1/wsf-mlops/src1007cced-i1030a/models/usecase2/runs/cd97jkwzjyqo/predictions/test/raw/predictions.csv",
    "messaging": "s3://src1007cced-243788878595-us-east-1/wsf-mlops/src1007cced-i1030a/models/usecase2/runs/cd97jkwzjyqo/predictions/test/raw/messaging.csv",
    "messaging_final": "s3://src1007cced-243788878595-us-east-1/wsf-mlops/src1007cced-i1030a/models/usecase2/runs/cd97jkwzjyqo/predictions/test/raw/messaging_final.csv"
}

# This checkout supplies the Python code you are editing, not the input data.
REPOSITORY = next(p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "mlops-models/src/mlops_models").is_dir())
sys.path.insert(0, str(REPOSITORY / "mlops-models/src"))

# Fresh local copies are needed because the step accepts filesystem paths.
work_directory = Path(mkdtemp(prefix="postprocess-test-"))
data_directory = work_directory / "data"
output_directory = work_directory / "output"
model_directory = work_directory / "model"
selection_file = work_directory / "selection/selected_features.csv"
prediction_directory = work_directory / "predictions"
parameter_path = model_directory / "parameters.json"
metadata_path = model_directory / "model_metadata.json"

output_directory.mkdir()
os.environ.update({
    "MODEL_ID": MODEL_ID,
    "SM_OUTPUT_DATA_DIR": str(output_directory),
    "SM_OUTPUT_DIR": str(output_directory),
    "MLOPS_PREDICTION_FORMAT": "csv",
    "MLOPS_TRACKING_URI": "",
    "MLOPS_WRITE_RUN_REPORT": "0",
    "MLOPS_LOG_ARTIFACTS": "0",
    "MLOPS_MODEL_DIR": str(model_directory),
    "SM_MODEL_DIR": str(model_directory),
})

# Download the exact files listed above without transforming them.
s3 = boto3.client("s3", region_name="us-east-1")
local_files = {
    'data': data_directory / "data.csv",
    'selection': selection_file,
    'model': work_directory / "model.tar.gz",
    'predictions': prediction_directory / "predictions.csv",
    'messaging': prediction_directory / "messaging.csv",
    'messaging_final': prediction_directory / "messaging_final.csv",
}
for name, local_file in local_files.items():
    local_file.parent.mkdir(parents=True, exist_ok=True)
    address = urlsplit(S3_INPUTS[name])
    s3.download_file(address.netloc, address.path.lstrip("/"), str(local_file))

# The archive supplies parameters.json, model_metadata.json and the fitted model.
with tarfile.open(local_files["model"]) as archive:
    archive.extractall(model_directory, filter="data")

parameters = json.loads(parameter_path.read_text())
metadata = json.loads(metadata_path.read_text())


In [ ]:
# Optional inspection; the following cell executes the step.
print("Downloaded from:", S3_INPUTS)
print("New local working directory:", work_directory)
print("Local experiment output:", output_directory)
display(pd.read_csv(data_directory / "data.csv", nrows=10))

display(pd.read_csv(selection_file), pd.read_csv(prediction_directory / "messaging_final.csv", nrows=10))


In [ ]:
from mlops_models.model_base.inference import postprocess_inference_test as postprocess

postprocess = importlib.reload(postprocess)
result_directory = postprocess.prediction_test(
    data_directory=data_directory,
    selection_file=selection_file,
    prediction_directory=prediction_directory,
    output_directory=output_directory,
    model_directory=model_directory,
)
result_directory
